In [1]:
# goal: for each question in our ground truth dataset, we run search - then we check whether the results include the correct document

In [1]:
from ingest import load_faq_data, build_index
import pandas as pd
from pprint import pprint
from tqdm.auto import tqdm

In [2]:
# read the ground truth data created
df_ground_truth = pd.read_csv(filepath_or_buffer='data/ground_truth_new.csv')
ground_truth = df_ground_truth.to_dict(orient='records')
pprint(ground_truth[:5])

[{'document': '74eb249bbf',
  'question': 'I just found out about this course. Is it too late for me to '
              'sign up?'},
 {'document': '74eb249bbf',
  'question': 'If I join the course now, can I still get a certificate? What '
              'do I need to do?'},
 {'document': '74eb249bbf',
  'question': 'Can I enter the course after it has already started?'},
 {'document': '74eb249bbf',
  'question': 'Is there a deadline for submitting the project if I want a '
              'certificate?'},
 {'document': '74eb249bbf',
  'question': 'What happens if I join late? Will I miss out on getting '
              'certified?'}]


In [3]:
# load the faq documents and index them
documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc['course'] == 'llm-zoomcamp':
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [4]:
# define the function to perform text search using the index created
def text_search(query):
    boost_dict = {'question': 3.0, 'section': 0.5}
    return index.search(query=query, num_results=5, boost_dict=boost_dict)

def compute_relevance_text(q):
    doc_id = q['document']
    results = text_search(query=q['question'])
    
    # turn the comparison into a relevance list - the index with value '1' specifies the the rank of the document in relevance comparison
    # (1 means the retrieved document has the same ID as the correct document) - relevance means whether a retrieved document is the 
    # correct document for this question
    relevance = []
    for d in results:
        # compare the retrieved document IDs with the correct document ID
        relevance.append(int(d['id'] == doc_id))

    return relevance

# perform query search for the first ground truth question and output the relevance list
q = ground_truth[0]
print(q['question'])
print(f'Relevance List from first ground truth: {compute_relevance_text(q)}') # correct document was found at the first position

I just found out about this course. Is it too late for me to sign up?
Relevance List from first ground truth: [1, 0, 0, 0, 0]


In [5]:
# calculate the relevance lists for all/more than one ground truth records
def compute_relevance_total_text(ground_truth):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance_text(q)
        relevance_total.append(relevance)

    return relevance_total

# test the relevance computation for 15 ground truth records
ground_truth_sample = ground_truth[:15]
relevance_total_text = compute_relevance_total_text(ground_truth_sample)
pprint(f'Relevance Lists: {relevance_total_text}')

  0%|          | 0/15 [00:00<?, ?it/s]

('Relevance Lists: [[1, 0, 0, 0, 0], [1, 0, 0, 0, 0], [0, 0, 0, 1, 0], [0, 0, '
 '0, 0, 0], [0, 1, 0, 0, 0], [1, 0, 0, 0, 0], [1, 0, 0, 0, 0], [1, 0, 0, 0, '
 '0], [1, 0, 0, 0, 0], [0, 0, 0, 1, 0], [1, 0, 0, 0, 0], [0, 1, 0, 0, 0], [0, '
 '1, 0, 0, 0], [1, 0, 0, 0, 0], [1, 0, 0, 0, 0]]')


In [6]:
# make the relevance functions generic - we may want to evaluate text search, vector search, hybrid search, or another retrieval method
# the relevance logic is the same - only the search function changes
def compute_relevance(q, search_function): # function to create the relevance list for one ground truth question
    # we pass the search function as an argument
    doc_id = q['document']
    results = search_function(query=q['question'])

    relevance = []
    for d in results:
        relevance.append(int(d['id'] == doc_id))

    return relevance

# function to create the relevance lists for more than one ground truth questions; we pass the search function as an argument
def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

# output the relevance lists for all the ground truth questions using text search; search results are represented as relevance lists
relevance_total = compute_relevance_total(ground_truth, text_search)

  0%|          | 0/565 [00:00<?, ?it/s]

In [7]:
# calculate hit rate
def hit_rate(relevance):
    cnt = 0
    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)

print(f'Hit Rate: {hit_rate(relevance=relevance_total)}')

Hit Rate: 0.8690265486725663


In [8]:
# calculate MRR
def mrr(relevance):
    total_score = 0.0

    for line in relevance: # a) [1,0,0,0,0], b) [0,0,1,0,0]
        for rank in range(len(line)): # rank - [0, 1, 2, 3, 4]
            if line[rank] == 1: # a) line[0] = 1, b) line[2] = 1
                total_score = total_score + 1 / (rank + 1) # a) total_score = (0 + 1) / (0 + 1) = 1
                # b) total_score = (1 + 1) / (2 + 1) = 0.66
                # we use (rank + 1) because Python counts positions from zero - the first position should score 1/1, and without the + 1 we'd divide by zero
                break
                 
    return total_score / len(relevance) # MRR is the average of these scores across all queries; it rewards systems that put the correct document near the top

print(f'MRR: {mrr(relevance=relevance_total)}')

MRR: 0.7365486725663714


In [9]:
# function that computes the relevance lists using a search function, and the search metrics
def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)
    
    return {
        'hit_rate': hit_rate(relevance_total),
        'mrr': mrr(relevance_total),
    }

print(evaluate(ground_truth=ground_truth, search_function=text_search))

  0%|          | 0/565 [00:00<?, ?it/s]

{'hit_rate': 0.8690265486725663, 'mrr': 0.7365486725663714}
